In [ ]:
import os
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain.agents.middleware import wrap_model_call
import json
from langchain.tools import tool
from pinecone import Pinecone
from langchain_pinecone import PineconeEmbeddings, PineconeVectorStore
from stockanalysis.yahoo_finance import get_latest_stock_price
from stockanalysis.utilities import (
    get_company_financials as fetch_company_financials,
    get_stock_history as fetch_stock_history,
    get_technical_indicators as fetch_technical_indicators,
)

In [ ]:
load_dotenv()

llm = ChatOpenAI(
    model=os.environ["NEBIUS_MODEL"],
    api_key=os.environ["NEBIUS_API_KEY"],
    base_url="https://api.tokenfactory.nebius.com/v1/",
    temperature=0,
    use_responses_api=False,
)

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index = pc.Index(os.environ["PINECONE_INDEX_NAME"])
print(f"Connected to Pinecone index: {index}")
embeddings = PineconeEmbeddings(
    model="llama-text-embed-v2",
    pinecone_api_key=os.environ["PINECONE_API_KEY"],
    query_params={
        "input_type": "query",
        "dimension": 512,
    },
    document_params={
        "input_type": "passage",
        "dimension": 512,
    },
)

vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace="stock-analysis-nsp",
)

print("Model:", llm.model_name)

In [ ]:
from langchain.tools import tool

@tool
def get_latest_price(symbol: str) -> dict:
    """Get the latest available stock price bar for a ticker."""
    return get_latest_stock_price(symbol)


@tool
def get_company_financials(symbol: str, frequency: str = "quarterly") -> dict:
    """Get up to four income, balance-sheet, and cash-flow periods.

    frequency must be quarterly or annual. Values are reported historical
    financials, not forecasts.
    """
    return fetch_company_financials(symbol, frequency)


@tool
def get_stock_history(symbol: str, period: str = "3mo") -> dict:
    """Get adjusted daily OHLCV history for a stock.

    period must be 1mo, 3mo, 6mo, 1y, or 2y.
    """
    return fetch_stock_history(symbol, period)


@tool
def get_technical_indicators(symbol: str) -> dict:
    """Calculate 20/50/200-day SMAs and 14-day RSI from adjusted prices."""
    return fetch_technical_indicators(symbol)



@tool
def search_stock_lessons(query: str) -> str:
    """Search the PDF lessons for stock analysis and trading concepts."""
    documents = vector_store.similarity_search(query, k=4)

    print(f"[RAG] Retrieved {len(documents)} documents", flush=True)

    if not documents:
        return "No relevant passages found."

    return json.dumps(
        [
            {
                "content": doc.page_content,
                "source": doc.metadata.get("source"),
                "title": doc.metadata.get("title"),
                "page_number": doc.metadata.get("page_number"),
            }
            for doc in documents
        ],
        ensure_ascii=False,
    )

In [ ]:
class StockState(TypedDict, total=False):
    question: str
    symbol: str
    horizon: str
    research_messages: list
    answer: str


def confirm_context(state: StockState):
    """Pause until the user confirms the research context."""
    response = interrupt({
        "message": "Confirm the ticker and investment horizon.",
        "proposed_symbol": state["symbol"],
        "proposed_horizon": state.get("horizon", ""),
    })

    symbol = str(response.get("symbol", "")).strip().upper()
    horizon = str(response.get("horizon", "")).strip()

    if not symbol or not horizon:
        raise ValueError("Provide both a symbol and an investment horizon.")

    return {
        "symbol": symbol,
        "horizon": horizon,
    }


def research(state: StockState):
    """Run the agent that has access to your tools."""
    result = research_agent.invoke({
        "messages": [{
            "role": "user",
            "content": (
                f"Research question: {state['question']}\n"
                f"Confirmed ticker: {state['symbol']}\n"
                f"Investment horizon: {state['horizon']}"
            ),
        }]
    })

    return {
        "research_messages": result["messages"],
        "answer": result["messages"][-1].content,
    }

In [ ]:
import importlib
from typing import TypedDict
from uuid import uuid4

from langchain.agents import create_agent
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

# Pick up saved changes to your Python file.


tools = [
    get_latest_price,
    search_stock_lessons,
    get_stock_history,
    get_technical_indicators,
    get_company_financials,
]

# THIS is where the tools are associated with your LLM.
# create_agent handles tool binding, execution, and returning results.
research_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You help users research stocks using the available tools. "
        "For current prices, call get_latest_price. "
        "For historical prices, call get_stock_history. "
        "For RSI and moving averages, call get_technical_indicators. "
        "For financial statements, call get_company_financials. "
        "For trading concepts, search stock lessons and cite available "
        "sources and page numbers. "
        "For a broad buy/sell question, fetch the latest price, "
        "technical indicators, and financial statements before answering. "
        "Use the confirmed ticker and investment horizon. "
        "Include timestamps, statement periods, and sources. "
        "Explain tool failures and missing data specifically. "
        "Do not invent prices, indicators, news, or forecasts. "
        "Do not claim tools are unavailable when they are registered. "
        "Web browsing is not provided by these tools."
    ),
)

In [ ]:
builder = StateGraph(StockState)

builder.add_node("confirm", confirm_context)
builder.add_node("research", research)

builder.add_edge(START, "confirm")
builder.add_edge("confirm", "research")
builder.add_edge("research", END)

# Retains state while this Python process stays alive.
checkpointer = InMemorySaver()

stock_graph = builder.compile(checkpointer=checkpointer)

In [ ]:
# Use a fresh ID for each independent research session.
config = {
    "configurable": {
        "thread_id": str(uuid4())
    }
}

result = stock_graph.invoke(
    {
        "question": "Can I buy Google stock now?",
        "symbol": "GOOGL",
        "horizon": "long term",
    },
    config=config,
)

# Display the question that paused the graph.
for pause in result.get("__interrupt__", []):
    print(pause.value)


result = stock_graph.invoke(
    Command(resume={
        "symbol": "GOOGL",
        "horizon": "long term — approximately 3 to 5 years",
    }),
    config=config,
)

print(result["answer"])

for message in result.get("research_messages", []):
    message.pretty_print()